# Case Study 1 — EXP-0 Nested Alpha Tuning (Development Only)

## Purpose

This notebook performs a **separate nested project-grouped sensitivity study**
for `EXP-0` regularisation (`sgd_alpha`). It is deliberately constrained to the
frozen 80% development partition.

It does **not** replace the already-completed EXP-0 baseline, does **not**
reopen formal final model selection, and **never loads or scores the frozen 20%
global outer holdout**.

### Nested design

```text
frozen 80% development data
└── frozen 5-fold project-grouped development evaluation
    ├── one development fold held out for final scoring
    └── remaining projects
        └── new 3-fold StratifiedGroupKFold inner CV
            └── choose alpha by pooled inner OOF PR-AUC
```

The global outer holdout remains untouched.

## Guardrails

- All three candidates use the original EXP-0 TF-IDF and SGD configuration.
- Only `sgd_alpha` is tuned over a predeclared grid.
- All TF-IDF fitting remains inside the relevant inner or outer training split.
- The inner profile scores only inner validation folds; it does **not** consume an outer-development test fold.
- The full run is resumable only from an interrupted run with the exact same stored configuration.
- Do not use this experiment to rerun the MLP holdout or change the locked main-study winner.

In [7]:
# ============================================================================
# 1. Google Drive and repository setup — actual repository layout
# ============================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "prashant"

# Git repository root
REPO_ROOT = Path("/content/DiverseVul--IS-Project")

# Actual Python project root inside the Git repository
PROJECT_DIR = REPO_ROOT / "vuln-detection"

# Actual source directory
SRC_DIR = PROJECT_DIR / "src"

BASE_EXP0_MODULE = (
    SRC_DIR / "case_study_1" / "exp0" / "exp0_lr.py"
)

NESTED_EXP0_MODULE = (
    SRC_DIR / "case_study_1" / "exp0" / "exp0_nested_alpha.py"
)


def run_command(command):
    """Run a command and stop clearly if it fails."""
    subprocess.run(command, check=True)


# Clone only if the repository does not already exist.
if not REPO_ROOT.is_dir():
    print("Cloning repository...")
    run_command(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_ROOT),
        ]
    )
else:
    print("Repository already present. Refreshing branch...")
    run_command(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
    run_command(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
    run_command(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH]
    )

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        "The Git repository was found, but the Python project folder is missing.\n\n"
        f"Expected project folder:\n{PROJECT_DIR}"
    )

if not SRC_DIR.is_dir():
    raise FileNotFoundError(
        "The project folder was found, but the source directory is missing.\n\n"
        f"Expected source folder:\n{SRC_DIR}"
    )

if not BASE_EXP0_MODULE.is_file():
    raise FileNotFoundError(
        "The project structure exists, but the canonical EXP-0 module is missing.\n\n"
        f"Expected:\n{BASE_EXP0_MODULE}"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nRepository setup successful.")
print("Git repository root:", REPO_ROOT)
print("Python project root:", PROJECT_DIR)
print("Source path:", SRC_DIR)
print("Base EXP-0 module:", BASE_EXP0_MODULE)

if NESTED_EXP0_MODULE.is_file():
    print("Nested EXP-0 module:", NESTED_EXP0_MODULE)
    print("\nNested-CV module is available. You may continue.")
else:
    print("\nNested EXP-0 module is NOT yet in GitHub:")
    print(" ", NESTED_EXP0_MODULE)
    print(
        "\nBefore running the nested-CV notebook, add and push "
        "exp0_nested_alpha.py to this exact location."
    )

run_command(["git", "-C", str(REPO_ROOT), "log", "-1", "--oneline"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository already present. Refreshing branch...

Repository setup successful.
Git repository root: /content/DiverseVul--IS-Project
Python project root: /content/DiverseVul--IS-Project/vuln-detection
Source path: /content/DiverseVul--IS-Project/vuln-detection/src
Base EXP-0 module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp0/exp0_lr.py
Nested EXP-0 module: /content/DiverseVul--IS-Project/vuln-detection/src/case_study_1/exp0/exp0_nested_alpha.py

Nested-CV module is available. You may continue.


In [8]:
# ============================================================================
# 2. Confirm runtime dependencies
# ============================================================================

import importlib.util

missing_packages = []
for package, import_name in [
    ('pyarrow', 'pyarrow'),
    ('scikit-learn', 'sklearn'),
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
]:
    if importlib.util.find_spec(import_name) is None:
        missing_packages.append(package)

if missing_packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages], check=True)

import numpy as np
import pandas as pd
import sklearn

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

numpy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1


In [9]:
# ============================================================================
# 3. Import canonical Case Study 1 code
# ============================================================================

import importlib

exp0_lr = importlib.import_module('case_study_1.exp0.exp0_lr')
exp0_nested = importlib.import_module('case_study_1.exp0.exp0_nested_alpha')
split_manifest = importlib.import_module('case_study_1.split_manifest')

required_module = SRC_DIR / 'case_study_1' / 'exp0' / 'exp0_nested_alpha.py'
if not required_module.is_file():
    raise FileNotFoundError(
        'Missing new nested EXP-0 module:\n'
        f'{required_module}\n\n'
        'Add the module from the provided package, commit it to branch prashant, '
        'then restart this notebook from a fresh runtime.'
    )

print('Base EXP-0 version:', exp0_lr.EXP0_VERSION)
print('Nested EXP-0 version:', exp0_nested.NESTED_EXP0_VERSION)

Base EXP-0 version: cs1-exp0-sgd-logistic-v2-holdout-innercv
Nested EXP-0 version: cs1-exp0-nested-alpha-v1-development-only


In [10]:
# ============================================================================
# 4. Frozen artifact paths — do not replace these manifests
# ============================================================================

DATA_ROOT = Path(
    '/content/drive/MyDrive/IntelligentSystemProject/'
    'VulnerabilityDetectionData'
)

PROCESSED_DIR = DATA_ROOT / 'processed'
MANIFEST_ROOT = DATA_ROOT / 'manifests' / 'cs1_project_holdout20_innercv_v1'
OUTPUT_ROOT = DATA_ROOT / 'outputs' / 'cs1_project_holdout20_innercv_v1'

NORMALIZED_DATASET_PATH = PROCESSED_DIR / 'rdiversevul_cs1_normalized_v1.parquet'
OUTER_MANIFEST_PATH = (
    MANIFEST_ROOT / 'outer_holdout' / 'cs1_outer_project_holdout_manifest.parquet'
)
INNER_MANIFEST_PATH = (
    MANIFEST_ROOT / 'inner_cv' / 'cs1_project_grouped_5fold_manifest.parquet'
)

NESTED_OUTPUT_DIR = OUTPUT_ROOT / 'exp0_nested_alpha_tuning_development_only_v1'

required_paths = {
    'normalized dataset': NORMALIZED_DATASET_PATH,
    'frozen outer holdout manifest': OUTER_MANIFEST_PATH,
    'frozen inner development-CV manifest': INNER_MANIFEST_PATH,
}

for label, path in required_paths.items():
    if not path.is_file():
        raise FileNotFoundError(f'Missing {label}:\n{path}')

print('Normalized dataset:', NORMALIZED_DATASET_PATH)
print('Frozen outer manifest:', OUTER_MANIFEST_PATH)
print('Frozen inner manifest:', INNER_MANIFEST_PATH)
print('Nested output directory:', NESTED_OUTPUT_DIR)

Normalized dataset: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
Frozen outer manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
Frozen inner manifest: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
Nested output directory: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/outputs/cs1_project_holdout20_innercv_v1/exp0_nested_alpha_tuning_development_only_v1


## Load only the frozen development partition

The outer manifest is used only to identify which rows belong to `development`.
The `outer_holdout` rows are explicitly excluded before any nested-CV code is
called.

In [11]:
# ============================================================================
# 5. Load frozen data and reconstruct the exact development partition
# ============================================================================

normalized_df = pd.read_parquet(NORMALIZED_DATASET_PATH)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)

inner_manifest_df = split_manifest.load_manifest(
    INNER_MANIFEST_PATH,
    config=split_manifest.SplitConfig(
        n_splits=5,
        random_state=42,
        shuffle=True,
    ),
)

required_outer_columns = {
    'source_row_id', 'label', 'project', 'partition', 'outer_holdout_fold'
}
missing_outer_columns = required_outer_columns - set(outer_manifest_df.columns)
if missing_outer_columns:
    raise KeyError(
        'Unexpected frozen outer-manifest schema. Missing: '
        f'{sorted(missing_outer_columns)}\n'
        f'Available: {outer_manifest_df.columns.tolist()}'
    )

if outer_manifest_df['source_row_id'].duplicated().any():
    raise RuntimeError('Frozen outer manifest has duplicate source_row_id values.')

expected_partitions = {'development', 'outer_holdout'}
actual_partitions = set(outer_manifest_df['partition'].astype(str).unique())
if actual_partitions != expected_partitions:
    raise RuntimeError(
        f'Unexpected partitions: {sorted(actual_partitions)}; '
        f'expected {sorted(expected_partitions)}.'
    )

base = normalized_df.merge(
    outer_manifest_df[
        ['source_row_id', 'label', 'project', 'partition', 'outer_holdout_fold']
    ],
    on='source_row_id',
    how='inner',
    validate='one_to_one',
    suffixes=('', '_outer'),
)

if len(base) != len(normalized_df) or len(base) != len(outer_manifest_df):
    raise RuntimeError(
        'Normalized dataset and frozen outer manifest do not have identical row coverage.\n'
        f'normalized={len(normalized_df):,}, manifest={len(outer_manifest_df):,}, merged={len(base):,}'
    )

if not (base['label'].astype(int) == base['label_outer'].astype(int)).all():
    raise RuntimeError('Label mismatch between normalized data and frozen outer manifest.')
if not (
    base['project'].astype(str).str.strip()
    == base['project_outer'].astype(str).str.strip()
).all():
    raise RuntimeError('Project mismatch between normalized data and frozen outer manifest.')

base = base.drop(columns=['label_outer', 'project_outer'])

development_df = (
    base.loc[base['partition'] == 'development']
    .drop(columns=['partition', 'outer_holdout_fold'])
    .reset_index(drop=True)
)

# Keep the holdout only as an ID set for a defensive no-access / no-overlap audit.
outer_holdout_ids = set(
    base.loc[base['partition'] == 'outer_holdout', 'source_row_id']
)

if set(development_df['source_row_id']).intersection(outer_holdout_ids):
    raise RuntimeError('Development rows overlap the frozen global outer holdout.')

if len(development_df) != 203_958:
    raise RuntimeError(
        f'Unexpected development size: {len(development_df):,}; expected 203,958.'
    )

print('All normalized rows: ', f'{len(base):,}')
print('Development rows:    ', f'{len(development_df):,}')
print('Holdout rows:        ', f'{len(outer_holdout_ids):,}')
print('Development positive rate:', f"{development_df['label'].mean():.6f}")

All normalized rows:  261,667
Development rows:     203,958
Holdout rows:         57,709
Development positive rate: 0.052594


In [12]:
# ============================================================================
# 6. Verify the frozen five-fold development manifest
# ============================================================================

inner_ids = set(inner_manifest_df['source_row_id'])
development_ids = set(development_df['source_row_id'])

if inner_ids != development_ids:
    raise RuntimeError(
        'The frozen five-fold development manifest does not match the reconstructed development partition.\n'
        f'Missing from manifest: {len(development_ids - inner_ids):,}\n'
        f'Extra in manifest:    {len(inner_ids - development_ids):,}'
    )

if inner_ids.intersection(outer_holdout_ids):
    raise RuntimeError('The development manifest unexpectedly contains global holdout rows.')

fold_audit = (
    inner_manifest_df.groupby('fold', as_index=False)
    .agg(
        rows=('source_row_id', 'size'),
        projects=('project', 'nunique'),
        vulnerable=('label', 'sum'),
        positive_rate=('label', 'mean'),
    )
    .sort_values('fold')
    .reset_index(drop=True)
)

display(fold_audit)

for fold_id in range(5):
    test_projects = set(inner_manifest_df.loc[inner_manifest_df['fold'] == fold_id, 'project'])
    train_projects = set(inner_manifest_df.loc[inner_manifest_df['fold'] != fold_id, 'project'])
    if test_projects.intersection(train_projects):
        raise RuntimeError(f'Project leakage in frozen development fold {fold_id}.')

print('Frozen development manifest verified: 5 folds, zero project leakage, zero global-holdout rows.')

,fold,rows,projects,vulnerable,positive_rate
0,0,55509,1,2420,0.043597
1,1,40791,158,2091,0.051261
2,2,39072,146,2062,0.052774
3,3,35016,137,2047,0.058459
4,4,33570,152,2107,0.062764


Frozen development manifest verified: 5 folds, zero project leakage, zero global-holdout rows.


## Fixed base pipeline and predeclared nested grid

This is intentionally a small sensitivity study. The only candidate setting is
`sgd_alpha`; all feature extraction and decision-policy choices stay fixed.

In [13]:
# ============================================================================
# 7. Declare the fixed EXP-0 base pipeline and nested alpha grid
# ============================================================================

BASE_EXP0_CONFIG = exp0_lr.Exp0Config(
    experiment_name='cs1_exp0_lr',
    code_column='normalized_code',
    source_id_column='source_row_id',
    label_column='label',
    project_column='project',
    fold_column='fold',
    n_splits=5,
    random_state=42,
    decision_threshold=0.50,
    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,
    char_analyzer='char',
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,
    lowercase=False,
    sublinear_tf=True,
    tfidf_norm='l2',
    sgd_loss='log_loss',
    sgd_penalty='l2',
    sgd_alpha=1e-5,  # baseline reference; nested code replaces only this per candidate
    sgd_class_weight='balanced',
    sgd_max_iter=80,
    sgd_tol=1e-3,
    sgd_average=True,
    top_features_per_direction=0,
    verbose=False,
)

NESTED_CONFIG = exp0_nested.NestedAlphaConfig(
    experiment_name='cs1_exp0_nested_alpha_dev_grouped',
    alpha_grid=(1e-6, 3e-6, 1e-5, 3e-5, 1e-4),
    inner_n_splits=3,
    inner_random_state=20260707,
    selection_metric='average_precision_pr_auc',
    decision_threshold=0.50,
    tie_break_rule='higher_alpha_then_grid_order',
    top_features_per_direction=0,
    verbose=True,
)

print('Base EXP-0 alpha (reference):', BASE_EXP0_CONFIG.sgd_alpha)
print('Nested alpha grid:', NESTED_CONFIG.alpha_grid)
print('Inner splitter:', f'StratifiedGroupKFold(n_splits={NESTED_CONFIG.inner_n_splits})')
print('Inner selection metric:', NESTED_CONFIG.selection_metric)
print('Decision threshold remains frozen at:', NESTED_CONFIG.decision_threshold)

Base EXP-0 alpha (reference): 1e-05
Nested alpha grid: (1e-06, 3e-06, 1e-05, 3e-05, 0.0001)
Inner splitter: StratifiedGroupKFold(n_splits=3)
Inner selection metric: average_precision_pr_auc
Decision threshold remains frozen at: 0.5


In [14]:
# ============================================================================
# 8. Provenance metadata for the new study
# ============================================================================

import hashlib


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

NESTED_METADATA = {
    'run_kind': 'nested_alpha_tuning_development_only',
    'scope_note': (
        'Post-selection robustness study. It does not replace the frozen EXP-0 baseline, '
        'reopen final main-study model selection, or access the global outer holdout.'
    ),
    'normalized_dataset_path': str(NORMALIZED_DATASET_PATH),
    'normalized_dataset_sha256': sha256_file(NORMALIZED_DATASET_PATH),
    'outer_manifest_path': str(OUTER_MANIFEST_PATH),
    'outer_manifest_sha256': sha256_file(OUTER_MANIFEST_PATH),
    'inner_manifest_path': str(INNER_MANIFEST_PATH),
    'inner_manifest_sha256': sha256_file(INNER_MANIFEST_PATH),
    'global_outer_holdout_used': False,
}

for key, value in NESTED_METADATA.items():
    print(f'{key}: {value}')

run_kind: nested_alpha_tuning_development_only
scope_note: Post-selection robustness study. It does not replace the frozen EXP-0 baseline, reopen final main-study model selection, or access the global outer holdout.
normalized_dataset_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_v1.parquet
normalized_dataset_sha256: e31c32fe85ac98af090e0d3fb51a067bfd6aa8473118d322042ea8b13086fc91
outer_manifest_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet
outer_manifest_sha256: 02ea91d43363cb094215ed0aee6102958800e318384a37a9bb82f0d28362342f
inner_manifest_path: /content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet
inner_manifest_sha256: fc9c364550b1f9028f3770062d1263e189f882e90f4a63

## Feasibility profile — inner tuning only

Set the first switch to `True` once. This runs the 3-fold **inner** tuning
process for one large outer-training partition, but it does not score the
corresponding outer-development test fold and does not touch the global holdout.

Do not change the grid after inspecting this profile.

In [18]:
# ============================================================================
# 9. Optional feasibility profile — no outer-development test score
# ============================================================================

RUN_NESTED_PROFILE = False
PROFILE_OUTER_DEVELOPMENT_FOLD = 4  # largest-ish outer training partition; zero-based fold ID

if RUN_NESTED_PROFILE:
    nested_profile = exp0_nested.run_exp0_nested_inner_profile(
        development_frame=development_df,
        development_manifest=inner_manifest_df,
        outer_fold_id=PROFILE_OUTER_DEVELOPMENT_FOLD,
        base_config=BASE_EXP0_CONFIG,
        nested_config=NESTED_CONFIG,
    )

    print('\nSelected alpha in profile:', nested_profile['selected_alpha']['selected_alpha'])
    print('Profile duration (minutes):', nested_profile['total_profile_seconds'] / 60.0)
    print('\nInner alpha summary:')
    display(nested_profile['alpha_summary'])
    print('\nInner split audit:')
    display(nested_profile['inner_split_audit'])
else:
    print(
        'Nested profile is disabled. Set RUN_NESTED_PROFILE = True only once for '
        'a computational feasibility check. It does not score an outer-development test fold.'
    )

Nested profile is disabled. Set RUN_NESTED_PROFILE = True only once for a computational feasibility check. It does not score an outer-development test fold.


## Official nested development-only run

Run only after reviewing the profile. This will take substantially longer than
the original EXP-0 baseline because every outer-development training partition
contains a new 3-fold project-grouped tuning stage. Checkpoints are written
after each completed outer-development fold.

- First run: `RUN_NESTED_OFFICIAL = True`, `RESUME_INTERRUPTED_RUN = False`
- After an interruption: `RUN_NESTED_OFFICIAL = True`, `RESUME_INTERRUPTED_RUN = True`
- Never set both `RUN_NESTED_PROFILE` and `RUN_NESTED_OFFICIAL` to `True`.

In [19]:
# ============================================================================
# 10. Guarded official nested run — DEVELOPMENT ONLY
# ============================================================================

RUN_NESTED_OFFICIAL = True
RESUME_INTERRUPTED_RUN = False

if RUN_NESTED_PROFILE and RUN_NESTED_OFFICIAL:
    raise RuntimeError(
        'Run either the feasibility profile or the official nested run, not both in one execution.'
    )

if RUN_NESTED_OFFICIAL:
    nested_results = exp0_nested.run_exp0_nested_alpha(
        development_frame=development_df,
        development_manifest=inner_manifest_df,
        base_config=BASE_EXP0_CONFIG,
        nested_config=NESTED_CONFIG,
        output_dir=NESTED_OUTPUT_DIR,
        resume=RESUME_INTERRUPTED_RUN,
        additional_metadata=NESTED_METADATA,
    )

    print('\nNested development-only run completed.')
    print('Output directory:', NESTED_OUTPUT_DIR)
    print('\nPooled nested OOF metrics:')
    display(
        pd.DataFrame(
            list(nested_results['evaluation']['pooled_metrics'].items()),
            columns=['metric', 'value'],
        )
    )
    print('\nSelected alpha by frozen development outer fold:')
    display(nested_results['selected_alpha'])
else:
    print(
        'Official nested run is disabled. Set RUN_NESTED_OFFICIAL = True only after '\
        'the profile has been reviewed. The global 20% outer holdout is never loaded.'
    )

[12:53:38] Nested EXP-0 alpha study started: 5 outer-development folds, 3 inner project-grouped folds, alpha grid=[1e-06, 3e-06, 1e-05, 3e-05, 0.0001].
[12:53:38] Scope: development partition only. The global 20% outer holdout is not loaded or scored.
[12:53:38] Outer development fold 1/5 | inner tuning started (3-fold project-grouped CV; 5 alpha values).
[12:53:39] Outer development fold 1/5, inner fold 1/3 | vectorizing once for all alpha candidates...
[12:59:10] Outer development fold 1/5, inner fold 1/3 | vectorization done in 5.46 min (110,000 features).
[13:00:07] Outer development fold 1/5, inner fold 2/3 | vectorizing once for all alpha candidates...
[13:05:30] Outer development fold 1/5, inner fold 2/3 | vectorization done in 5.37 min (110,000 features).
[13:06:19] Outer development fold 1/5, inner fold 3/3 | vectorizing once for all alpha candidates...
[13:11:38] Outer development fold 1/5, inner fold 3/3 | vectorization done in 5.30 min (110,000 features).
[13:12:32] Outer d

,metric,value
0,n_samples,203958.000000
1,vulnerable_1,10727.000000
2,non_vulnerable_0,193231.000000
3,positive_rate,0.052594
4,threshold,0.500000
5,average_precision_pr_auc,0.141971
6,precision,0.123763
7,recall,0.531836
8,f1,0.200799
9,mcc,0.172285



Selected alpha by frozen development outer fold:


,outer_fold,selected_alpha,selected_inner_pooled_pr_auc,selected_inner_mean_fold_pr_auc,selection_metric,tie_break_rule,total_inner_vectorization_seconds,outer_train_rows,outer_train_projects,alpha_grid,inner_n_splits
0,0,0.0001,0.158335,0.159439,average_precision_pr_auc,higher_alpha_then_grid_order,967.893928,148449,593,1e-06;3e-06;1e-05;3e-05;0.0001,3
1,1,0.0001,0.141972,0.140873,average_precision_pr_auc,higher_alpha_then_grid_order,995.968555,163167,436,1e-06;3e-06;1e-05;3e-05;0.0001,3
2,2,0.0001,0.136127,0.138791,average_precision_pr_auc,higher_alpha_then_grid_order,1006.170100,164886,448,1e-06;3e-06;1e-05;3e-05;0.0001,3
3,3,0.0001,0.135918,0.138022,average_precision_pr_auc,higher_alpha_then_grid_order,990.037855,168942,457,1e-06;3e-06;1e-05;3e-05;0.0001,3
4,4,0.0001,0.132316,0.143156,average_precision_pr_auc,higher_alpha_then_grid_order,1017.898276,170388,442,1e-06;3e-06;1e-05;3e-05;0.0001,3


## Read-only review of saved nested artifacts

This cell must only be run after the official nested study finishes. It loads
results; it does not train, tune, or score a model.

In [20]:
# ============================================================================
# 11. Read-only review of completed nested artifacts
# ============================================================================

NESTED_EXPERIMENT_NAME = NESTED_CONFIG.experiment_name
POOLED_METRICS_PATH = NESTED_OUTPUT_DIR / f'{NESTED_EXPERIMENT_NAME}_pooled_metrics.json'
SELECTED_ALPHA_PATH = NESTED_OUTPUT_DIR / f'{NESTED_EXPERIMENT_NAME}_selected_alpha_per_outer_fold.csv'
INNER_SCORES_PATH = NESTED_OUTPUT_DIR / f'{NESTED_EXPERIMENT_NAME}_inner_alpha_scores.csv'
INNER_AUDIT_PATH = NESTED_OUTPUT_DIR / f'{NESTED_EXPERIMENT_NAME}_inner_split_audit.csv'
OUTER_TRAINING_PATH = NESTED_OUTPUT_DIR / f'{NESTED_EXPERIMENT_NAME}_outer_fold_training.csv'
RUN_METADATA_PATH = NESTED_OUTPUT_DIR / f'{NESTED_EXPERIMENT_NAME}_run_metadata.json'

review_paths = {
    'pooled metrics': POOLED_METRICS_PATH,
    'selected alpha': SELECTED_ALPHA_PATH,
    'inner scores': INNER_SCORES_PATH,
    'inner audit': INNER_AUDIT_PATH,
    'outer training': OUTER_TRAINING_PATH,
    'run metadata': RUN_METADATA_PATH,
}

missing = {name: path for name, path in review_paths.items() if not path.is_file()}
if missing:
    print('Nested study is not complete yet. Missing final artifacts:')
    for name, path in missing.items():
        print(f' - {name}: {path}')
else:
    import json

    with POOLED_METRICS_PATH.open('r', encoding='utf-8') as file:
        nested_pooled_metrics = json.load(file)
    with RUN_METADATA_PATH.open('r', encoding='utf-8') as file:
        nested_run_metadata = json.load(file)

    saved_selected_alpha = pd.read_csv(SELECTED_ALPHA_PATH)
    saved_inner_scores = pd.read_csv(INNER_SCORES_PATH)
    saved_inner_audit = pd.read_csv(INNER_AUDIT_PATH)
    saved_outer_training = pd.read_csv(OUTER_TRAINING_PATH)

    if nested_run_metadata.get('global_outer_holdout_used') is not False:
        raise RuntimeError('Safety violation: nested metadata does not confirm global holdout exclusion.')

    if int(nested_pooled_metrics['n_samples']) != len(development_df):
        raise RuntimeError('Nested OOF row count does not match frozen development partition.')

    print('Nested study artifacts verified.')
    print('Global outer holdout used:', nested_run_metadata['global_outer_holdout_used'])
    print('\nPooled nested OOF metrics:')
    display(pd.DataFrame(list(nested_pooled_metrics.items()), columns=['metric', 'value']))
    print('\nAlpha selected independently for each outer-development fold:')
    display(saved_selected_alpha)
    print('\nInner split integrity summary:')
    display(saved_inner_audit)
    print('\nOuter refit metadata:')
    display(saved_outer_training)

Nested study artifacts verified.
Global outer holdout used: False

Pooled nested OOF metrics:


,metric,value
0,n_samples,203958.000000
1,vulnerable_1,10727.000000
2,non_vulnerable_0,193231.000000
3,positive_rate,0.052594
4,threshold,0.500000
5,average_precision_pr_auc,0.141971
6,precision,0.123763
7,recall,0.531836
8,f1,0.200799
9,mcc,0.172285



Alpha selected independently for each outer-development fold:


,outer_fold,selected_alpha,selected_inner_pooled_pr_auc,selected_inner_mean_fold_pr_auc,selection_metric,tie_break_rule,total_inner_vectorization_seconds,outer_train_rows,outer_train_projects,alpha_grid,inner_n_splits
0,0,0.0001,0.158335,0.159439,average_precision_pr_auc,higher_alpha_then_grid_order,967.893928,148449,593,1e-06;3e-06;1e-05;3e-05;0.0001,3
1,1,0.0001,0.141972,0.140873,average_precision_pr_auc,higher_alpha_then_grid_order,995.968555,163167,436,1e-06;3e-06;1e-05;3e-05;0.0001,3
2,2,0.0001,0.136127,0.138791,average_precision_pr_auc,higher_alpha_then_grid_order,1006.170100,164886,448,1e-06;3e-06;1e-05;3e-05;0.0001,3
3,3,0.0001,0.135918,0.138022,average_precision_pr_auc,higher_alpha_then_grid_order,990.037855,168942,457,1e-06;3e-06;1e-05;3e-05;0.0001,3
4,4,0.0001,0.132316,0.143156,average_precision_pr_auc,higher_alpha_then_grid_order,1017.898276,170388,442,1e-06;3e-06;1e-05;3e-05;0.0001,3



Inner split integrity summary:


,outer_fold,inner_fold,inner_train_rows,inner_validation_rows,inner_train_projects,inner_validation_projects,inner_train_positive_rate,inner_validation_positive_rate,inner_project_overlap,word_tfidf_seconds,char_tfidf_seconds,sparse_join_seconds,vectorization_seconds,word_features,char_features,total_features,train_nonzero_entries,test_nonzero_entries,train_matrix_density,test_matrix_density
0,0,0,102148,46301,398,195,0.053491,0.061403,0,111.618800,214.540269,1.339441,327.498510,50000,60000,110000,74480481,31354347,0.006629,0.006156
1,0,1,90912,57537,397,196,0.059871,0.049777,0,96.594011,223.984949,1.527921,322.106881,50000,60000,110000,66247615,39264733,0.006625,0.006204
2,0,2,103838,44611,391,202,0.054961,0.058282,0,104.341164,212.779678,1.167694,318.288537,50000,60000,110000,75110912,31181778,0.006576,0.006354
3,1,0,122593,40574,251,185,0.049097,0.064499,0,112.520423,232.695462,1.285082,346.500968,50000,60000,110000,86878110,29216004,0.006442,0.006546
4,1,1,57584,105583,252,184,0.059409,0.049392,0,78.797285,219.871951,1.106728,299.775965,50000,60000,110000,45221003,68178556,0.007139,0.005870
5,1,2,146157,17010,369,67,0.053586,0.047266,0,125.733566,222.732779,1.225278,349.691623,50000,60000,110000,103892674,12742460,0.006462,0.006810
6,2,0,146557,18329,388,60,0.053576,0.044356,0,125.005581,219.623886,1.121470,345.750938,50000,60000,110000,105536237,11843452,0.006546,0.005874
7,2,1,128826,36060,266,182,0.051224,0.057293,0,121.542321,223.883354,1.393615,346.819290,50000,60000,110000,92431613,24472612,0.006523,0.006170
8,2,2,54389,110497,242,206,0.052933,0.052363,0,74.889195,236.522099,2.188578,313.599872,50000,60000,110000,39706669,73933643,0.006637,0.006083
9,3,0,89017,79925,389,68,0.054293,0.048133,0,98.401318,217.994519,2.254518,318.650355,50000,60000,110000,65770633,52544432,0.006717,0.005977



Outer refit metadata:


,outer_fold,selected_alpha,outer_train_rows,outer_test_rows,outer_train_vulnerable,outer_test_vulnerable,outer_train_positive_rate,outer_test_positive_rate,outer_train_projects,outer_test_projects,...,test_matrix_density,model_fit_seconds,prediction_seconds,model_n_iter,convergence_warning_count,convergence_warning_messages,outer_total_fold_seconds,outer_score_min,outer_score_max,outer_score_mean
0,0,0.0001,148449,55509,8307,2420,0.055959,0.043597,593,1,...,0.005934,10.649892,0.085920,12,0,NaN,455.477365,0.035157,0.946235,0.323312
1,1,0.0001,163167,40791,8636,2091,0.052927,0.051261,436,158,...,0.006090,12.703096,0.067560,14,0,NaN,445.557970,0.025377,0.953335,0.364725
2,2,0.0001,164886,39072,8665,2062,0.052551,0.052774,448,146,...,0.006101,13.069444,0.062587,14,0,NaN,447.926166,0.034571,0.963451,0.391188
3,3,0.0001,168942,35016,8680,2047,0.051379,0.058459,457,137,...,0.006092,17.709765,0.059603,19,0,NaN,462.146061,0.018680,0.963396,0.362665
4,4,0.0001,170388,33570,8620,2107,0.050590,0.062764,442,152,...,0.006713,15.881892,0.068798,17,0,NaN,459.394571,0.039834,0.968028,0.395021


## Reporting boundary

After completion, describe this as a **nested-CV robustness/sensitivity study**
for EXP-0 alpha. It provides a more conservative estimate of a tuning
procedure, but it does not retroactively replace the locked main-study winner
or permit another global-holdout evaluation.